# Chunking & Embedding
**Purpose:** Transform cleaned event data into vector embeddings 
ready for FAISS indexing. Given that descriptions are very short 
(median 50 chars), each event is embedded as a single enriched 
text unit — no recursive splitting needed.

**Input:** `data/raw/events_paris.json`  
**Output:** 
- `data/processed/chunks.json` — text chunks with metadata  
- `data/processed/embeddings.npy` — numpy array of vectors  

**Author:** Hope  
**Date:** 2026-03-23

## 1. Imports and environment validation

In [1]:
# Cell 1 — Imports and environment validation
import json
import time
import numpy as np
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv
import os
import pandas as pd
from mistralai import Mistral

load_dotenv()
load_dotenv(dotenv_path=Path("../.env"))
api_key = os.getenv("MISTRAL_API_KEY")
assert api_key is not None, "MISTRAL_API_KEY not found in .env"

client = Mistral(api_key=api_key)

print("Environment loaded successfully.")
print(f"Mistral API key loaded: {'*' * 20}{api_key[-4:]}")
print(f"Mistral client initialised: {type(client)}")


Environment loaded successfully.
Mistral API key loaded: ********************L0qL
Mistral client initialised: <class 'mistralai.sdk.Mistral'>


In [2]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
print(".env exists here:", Path(".env").exists())
print(".env exists one level up:", Path("../.env").exists())

Current working directory: /Users/hopedonglo/Documents/Projects/OpenClassRooms/OpenClassroom_Projects/project_11/notebooks
.env exists here: False
.env exists one level up: True


In [3]:
# Cell 1 — Imports and environment validation
import json
import time
import numpy as np
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv
import os
import pandas as pd
from mistralai import Mistral

# Explicitly point to .env in project root
api_key = os.getenv("MISTRAL_API_KEY")
assert api_key is not None, "MISTRAL_API_KEY not found in .env"

client = Mistral(api_key=api_key)

print("Environment loaded successfully.")
print(f"Mistral API key loaded: {'*' * 20}{api_key[-4:]}")
print(f"Mistral client initialised: {type(client)}")

Environment loaded successfully.
Mistral API key loaded: ********************L0qL
Mistral client initialised: <class 'mistralai.sdk.Mistral'>


## 1. Configuration

In [4]:
INPUT_PATH  = Path("../data/raw/events_paris.json")
OUTPUT_DIR  = Path("../data/processed")
CHUNKS_PATH = OUTPUT_DIR / "chunks.json"
EMBED_PATH  = OUTPUT_DIR / "embeddings.npy"

EMBEDDING_MODEL = "mistral-embed"
BATCH_SIZE      = 10   # embeddings per API call
SLEEP_BETWEEN   = 0.5  # seconds between batches

print("Configuration set:")
print(f"  Input       : {INPUT_PATH}")
print(f"  Chunks out  : {CHUNKS_PATH}")
print(f"  Embeddings  : {EMBED_PATH}")
print(f"  Model       : {EMBEDDING_MODEL}")
print(f"  Batch size  : {BATCH_SIZE}")

Configuration set:
  Input       : ../data/raw/events_paris.json
  Chunks out  : ../data/processed/chunks.json
  Embeddings  : ../data/processed/embeddings.npy
  Model       : mistral-embed
  Batch size  : 10


## 

## 1.2  load and inspect the cleaned data:

In [5]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    events = json.load(f)

df = pd.DataFrame(events)

print(f"Events loaded : {len(df)}")
print(f"Columns       : {df.columns.tolist()}")
print(f"\nSample title  : {df['title.fr'].iloc[0]}")
print(f"Sample desc   : {df['description.fr'].iloc[0]}")

Events loaded : 1643
Columns       : ['uid', 'title.fr', 'description.fr', 'location.city', 'location.address', 'location.name', 'firstTiming.begin', 'lastTiming.end', 'dateRange.fr', 'slug', 'attendanceMode', 'categories', 'types-devenement']

Sample title  : Nouvel envol
Sample desc   : Emmanuelle Neuville, assistante paroissiale, quittera la paroisse après plusieurs années à son service fin septembre début octobre.


## 1.3 Build text chunks

In [6]:
# Strategy: concatenate title + description into one enriched text unit per event
# Rationale: descriptions are very short (median 50 chars) — no splitting needed

def build_chunk_text(row):
    """Combine title and description into a single rich text unit."""
    title       = row["title.fr"] or ""
    description = row["description.fr"] or ""
    city        = row["location.city"] or ""
    venue       = row["location.name"] or ""
    date        = row["dateRange.fr"] or ""

    return (
        f"Événement : {title}\n"
        f"Description : {description}\n"
        f"Lieu : {venue}, {city}\n"
        f"Date : {date}"
    )

# Build chunks with metadata
chunks = []
for _, row in df.iterrows():
    chunk = {
        "uid"         : row["uid"],
        "text"        : build_chunk_text(row),
        "title"       : row["title.fr"],
        "city"        : row["location.city"],
        "address"     : row["location.address"],
        "venue"       : row["location.name"],
        "date_begin"  : row["firstTiming.begin"],
        "date_end"    : row["lastTiming.end"],
        "date_range"  : row["dateRange.fr"],
        "slug"        : row["slug"],
        "url"         : f"https://openagenda.com/deciding-for-paris/events/{row['slug']}"
    }
    chunks.append(chunk)

print(f"Chunks built  : {len(chunks)}")
print(f"\nSample chunk text:\n{chunks[0]['text']}")
print(f"\nSample URL: {chunks[0]['url']}")

Chunks built  : 1643

Sample chunk text:
Événement : Nouvel envol
Description : Emmanuelle Neuville, assistante paroissiale, quittera la paroisse après plusieurs années à son service fin septembre début octobre.
Lieu : paroisse Saint-Jean-Baptiste-de-la-Salle, Paris
Date : 29 juin - 5 octobre

Sample URL: https://openagenda.com/deciding-for-paris/events/nouvel-envol


## 1.4 Test embedding on single chunk

In [7]:
# Always test one before committing to 1,740 API calls

def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed a batch of texts using Mistral embedding model."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        inputs=texts
    )
    return [item.embedding for item in response.data]

# Test on first chunk
test_embedding = embed_texts([chunks[0]["text"]])

print(f"Embedding dimension : {len(test_embedding[0])}")
print(f"First 5 values      : {test_embedding[0][:5]}")
print(f"Type                : {type(test_embedding[0][0])}")

Embedding dimension : 1024
First 5 values      : [-0.0297393798828125, 0.0189361572265625, 0.051910400390625, 0.004840850830078125, 0.032684326171875]
Type                : <class 'float'>


## 1.4 Embed all chunks in batches

In [8]:
# Cell 7 — Robust embedding with automatic rate limit recovery
# Reset and start fresh with safer parameters

all_embeddings  = []
BATCH_SIZE_SAFE = 5    # smaller batches
SLEEP_SAFE      = 3.0  # longer sleep between batches
SLEEP_429       = 60   # wait time on rate limit hit

total_batches = (len(chunks) + BATCH_SIZE_SAFE - 1) // BATCH_SIZE_SAFE
print(f"Total batches to process: {total_batches}")


for i in range(0, len(chunks), BATCH_SIZE_SAFE):
    batch       = chunks[i : i + BATCH_SIZE_SAFE]
    batch_texts = [c["text"] for c in batch]
    batch_num   = (i // BATCH_SIZE_SAFE) + 1

    try:
        embeddings = embed_texts(batch_texts)
        all_embeddings.extend(embeddings)

        if batch_num % 50 == 0 or batch_num == total_batches:
            print(f"  Batch {batch_num}/{total_batches} — "
                  f"cumulative: {len(all_embeddings)}/{len(chunks)}")

    except Exception as e:
        if "429" in str(e):
            print(f"\n  Rate limit hit at batch {batch_num} "
                  f"(chunk {i}, {len(all_embeddings)} embedded).")
            print(f"  Waiting {SLEEP_429}s before retrying...")
            time.sleep(SLEEP_429)
            # Retry same batch
            try:
                embeddings = embed_texts(batch_texts)
                all_embeddings.extend(embeddings)
                print(f"  Retry successful — continuing.")
            except Exception as retry_e:
                print(f"  Retry failed: {retry_e}")
                print(f"  Stopping at {len(all_embeddings)} embeddings.")
                break
        else:
            print(f"\n  ERROR at batch {batch_num} (chunk {i}):")
            print(f"  {type(e).__name__}: {e}")
            print(f"  Stopping at {len(all_embeddings)} embeddings.")
            break

    time.sleep(SLEEP_SAFE)

print(f"\nFinal count : {len(all_embeddings)}/{len(chunks)} embeddings")
assert len(all_embeddings) == len(chunks), \
    f"Mismatch: {len(all_embeddings)} embeddings vs {len(chunks)} chunks"
print("Assertion passed — all chunks embedded successfully.")

Total batches to process: 329
  Batch 50/329 — cumulative: 250/1643


KeyboardInterrupt: 

## 1.5 Save chunks and embeddings to disk

In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save chunks metadata as JSON
with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print(f"Chunks saved    : {CHUNKS_PATH}")

# Save embeddings as numpy array
embeddings_array = np.array(all_embeddings, dtype=np.float32)
np.save(EMBED_PATH, embeddings_array)
print(f"Embeddings saved: {EMBED_PATH}")

# Verify
print(f"\nChunks shape    : {len(chunks)} chunks")
print(f"Embeddings shape: {embeddings_array.shape}")
print(f"Dtype           : {embeddings_array.dtype}")

Chunks saved    : ../data/processed/chunks.json
Embeddings saved: ../data/processed/embeddings.npy

Chunks shape    : 1740 chunks
Embeddings shape: (1740, 1024)
Dtype           : float32
